In [1]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/249de2a1bc6c07516904de4a27993d56eaa9e2ad/01_data/processed/df_cleaned_ver2.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [2]:
# ============================================================
# CELL 1: DATA PREPARATION
# TF-IDF + SENTIMENT ANALYSIS + SMOTE
# ============================================================

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from scipy.sparse import hstack
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# 1. TEXT, SENTIMENT FEATURES AND TARGET
# ------------------------------------------------------------

X_text = df['lemmatized_tweet'].fillna('')
# Sentiment Analysis features
sentiment_features = [
    'negative',
    'positive',
    'neutral',
    'compound'
]

X_sentiment = df[sentiment_features].fillna(0)

# Target variable
y = df['cyberbullying_type']

# ------------------------------------------------------------
# 2. LABEL ENCODING
# ------------------------------------------------------------

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)

# Label mapping
label_mapping = dict(
    enumerate(label_encoder.classes_)
)

print("\nLabel Mapping:")

for number, category in label_mapping.items():
    print(f"{number} → {category}")

# ------------------------------------------------------------
# 3. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_text_train, X_text_test, X_sent_train, X_sent_test, y_train, y_test = train_test_split(
    X_text,
    X_sentiment,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain-Test Split")
print("-------------------------")
print("Training samples:", len(X_text_train))
print("Testing samples :", len(X_text_test))

# ------------------------------------------------------------
# 4. TF-IDF VECTORIZATION
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

print("\nTF-IDF Features")
print("-------------------------")
print("X_train TF-IDF:", X_train_tfidf.shape)
print("X_test TF-IDF :", X_test_tfidf.shape)

# ------------------------------------------------------------
# 5. COMBINE TF-IDF + SENTIMENT FEATURES
# ------------------------------------------------------------

X_train_combined = hstack([
    X_train_tfidf,
    X_sent_train.values
])

X_test_combined = hstack([
    X_test_tfidf,
    X_sent_test.values
])

print("\nCombined Features")
print("-------------------------")
print("X_train:", X_train_combined.shape)
print("X_test :", X_test_combined.shape)

Classes:
['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Label Mapping:
0 → age
1 → ethnicity
2 → gender
3 → not_cyberbullying
4 → other_cyberbullying
5 → religion

Train-Test Split
-------------------------
Training samples: 29821
Testing samples : 12781

TF-IDF Features
-------------------------
X_train TF-IDF: (29821, 60000)
X_test TF-IDF : (12781, 60000)

Combined Features
-------------------------
X_train: (29821, 60004)
X_test : (12781, 60004)


In [3]:
# ============================================================
# EXPERIMENT E: 10K TF-IDF FEATURES + SENTIMENT ANALYSIS
# NO FEATURE SELECTION
# INDEPENDENT VARIABLES — DOES NOT OVERWRITE OTHER CELLS
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# ------------------------------------------------------------
# 1. EXPERIMENT E — TF-IDF VECTORIZER
# ------------------------------------------------------------

tfidf_E = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

# Fit only on training text; transform test text
X_train_E_tfidf = tfidf_E.fit_transform(X_text_train)
X_test_E_tfidf = tfidf_E.transform(X_text_test)

print("Experiment E — 10K TF-IDF Features")
print("----------------------------------")
print("TF-IDF train shape:", X_train_E_tfidf.shape)
print("TF-IDF test shape :", X_test_E_tfidf.shape)

# ------------------------------------------------------------
# 2. COMBINE TF-IDF + SENTIMENT FEATURES
# ------------------------------------------------------------

X_train_E_combined = hstack([
    X_train_E_tfidf,
    csr_matrix(X_sent_train.to_numpy(dtype=float))
], format='csr')

X_test_E_combined = hstack([
    X_test_E_tfidf,
    csr_matrix(X_sent_test.to_numpy(dtype=float))
], format='csr')

# ------------------------------------------------------------
# 3. VERIFY FINAL FEATURE MATRICES
# ------------------------------------------------------------

print("\nExperiment E — Final Combined Features")
print("--------------------------------------")
print("Training shape:", X_train_E_combined.shape)
print("Testing shape :", X_test_E_combined.shape)

print("\nConfiguration:")
print("TF-IDF max_features: 10,000")
print("N-gram range       : (1, 2)")
print("Sentiment features : negative, positive, neutral, compound")
print("Feature selection  : None")
print("Train-test split   : Reuses the existing 70:30 split")
print("Random state       : 42 (from the existing split)")

Experiment E — 10K TF-IDF Features
----------------------------------
TF-IDF train shape: (29821, 10000)
TF-IDF test shape : (12781, 10000)

Experiment E — Final Combined Features
--------------------------------------
Training shape: (29821, 10004)
Testing shape : (12781, 10004)

Configuration:
TF-IDF max_features: 10,000
N-gram range       : (1, 2)
Sentiment features : negative, positive, neutral, compound
Feature selection  : None
Train-test split   : Reuses the existing 70:30 split
Random state       : 42 (from the existing split)


In [4]:
# ============================================================
# CELL 2: FEATURE SELECTION SETUP
# ============================================================

from sklearn.feature_selection import SelectKBest, chi2
from scipy.sparse import csr_matrix, hstack
import numpy as np

# Number of TEXT features to retain
K = 10000

print("Original TF-IDF features:", X_train_tfidf.shape[1])
print("Selected TF-IDF features :", K)

Original TF-IDF features: 60000
Selected TF-IDF features : 10000


In [5]:
# ============================================================
# CELL 3: EXPERIMENT A — NO FEATURE SELECTION
# ============================================================

X_train_A = X_train_tfidf
X_test_A = X_test_tfidf

print("Experiment A — No Selection")
print("----------------------------")
print("Train shape:", X_train_A.shape)
print("Test shape :", X_test_A.shape)

Experiment A — No Selection
----------------------------
Train shape: (29821, 60000)
Test shape : (12781, 60000)


In [6]:
X_train_A_combined = hstack([
    X_train_A,
    csr_matrix(X_sent_train.values)
])

X_test_A_combined = hstack([
    X_test_A,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_A_combined.shape)
print("Final test shape :", X_test_A_combined.shape)

Final train shape: (29821, 60004)
Final test shape : (12781, 60004)


In [7]:
# ============================================================
# CELL 4: EXPERIMENT B — GLOBAL FEATURE SELECTION
# ============================================================

global_selector = SelectKBest(
    score_func=chi2,
    k=K
)

# IMPORTANT:
# Fit only on training data
X_train_B = global_selector.fit_transform(
    X_train_tfidf,
    y_train
)

# Apply the SAME selector to test data
X_test_B = global_selector.transform(
    X_test_tfidf
)

print("Experiment B — Global Selection")
print("--------------------------------")
print("Train shape:", X_train_B.shape)
print("Test shape :", X_test_B.shape)

Experiment B — Global Selection
--------------------------------
Train shape: (29821, 10000)
Test shape : (12781, 10000)


In [8]:
X_train_B_combined = hstack([
    X_train_B,
    csr_matrix(X_sent_train.values)
])

X_test_B_combined = hstack([
    X_test_B,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_B_combined.shape)
print("Final test shape :", X_test_B_combined.shape)

Final train shape: (29821, 10004)
Final test shape : (12781, 10004)


In [9]:
# ============================================================
# CELL 5: TOP GLOBAL FEATURES
# ============================================================

feature_names = np.array(tfidf.get_feature_names_out())

global_selected_features = feature_names[
    global_selector.get_support()
]

print("Number of selected features:",
      len(global_selected_features))

print("\nFirst 100 selected features:")
print(global_selected_features[:100])

Number of selected features: 10000

First 100 selected features:
['aap' 'abdul' 'abdul kalam' 'able bully' 'able help' 'abortion'
 'abortion joke' 'abortion provider' 'abortion rape' 'absolute'
 'absolute idiot' 'absolutely' 'absolutely fucking' 'absorbed' 'abt girl'
 'abt rape' 'abu' 'abuse' 'abuse bullying' 'abuse gay' 'abuse happens'
 'abuse joke' 'abuse policy' 'abuse rape' 'abuse social' 'abuse woman'
 'abusive' 'abusive bully' 'acc' 'accept' 'accept one' 'accept radical'
 'accept truth' 'acceptable' 'accepts' 'accost' 'accost female' 'account'
 'account sure' 'acct' 'accts' 'accurate' 'ache' 'achieve' 'acne'
 'acquittal' 'acquittal christian' 'act' 'act dumb' 'act gay' 'act like'
 'acted' 'acted like' 'acting' 'acting dumb' 'acting like' 'acting muslim'
 'action' 'action idiot' 'active member' 'activity' 'actor' 'actual fuck'
 'actual muslim' 'actually black' 'actually bullied' 'actually dumb'
 'actually want' 'ad break' 'ad pro' 'adam baldwin' 'add facebook'
 'added video' 'addr

In [10]:
# ============================================================
# CELL 6: EXPERIMENT C — CLASS-AWARE FEATURE SELECTION
# ============================================================

n_classes = len(label_encoder.classes_)

# Number of features we want to finally retain
K_CLASS_AWARE = 10000

# Store Chi-square scores for every class
class_scores = np.zeros(
    (n_classes, X_train_tfidf.shape[1])
)

print("Classes:")
print(label_encoder.classes_)

print("\nCalculating class-specific feature scores...")

Classes:
['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Calculating class-specific feature scores...


In [11]:
# ============================================================
# ONE-VS-REST CHI-SQUARE SCORING
# ============================================================

for class_id in range(n_classes):

    # Current class = 1
    # All other classes = 0
    y_binary = (y_train == class_id).astype(int)

    scores, p_values = chi2(
        X_train_tfidf,
        y_binary
    )

    class_scores[class_id] = scores

    print(
        f"Class {class_id}: "
        f"{label_encoder.classes_[class_id]}"
    )

print("\nClass-aware scoring completed.")

Class 0: age
Class 1: ethnicity
Class 2: gender
Class 3: not_cyberbullying
Class 4: other_cyberbullying
Class 5: religion

Class-aware scoring completed.


In [12]:
# ============================================================
# CELL 7: CLASS-AWARE FEATURE RANKING
# ============================================================

# For each feature, find its strongest class-specific score
class_aware_scores = np.max(
    class_scores,
    axis=0
)

# Select top K features
class_aware_indices = np.argsort(
    class_aware_scores
)[::-1][:K_CLASS_AWARE]

# Boolean mask
class_aware_mask = np.zeros(
    X_train_tfidf.shape[1],
    dtype=bool
)

class_aware_mask[class_aware_indices] = True

print("Class-aware selected features:",
      class_aware_mask.sum())

Class-aware selected features: 10000


In [13]:
X_train_C = X_train_tfidf[:, class_aware_mask]
X_test_C = X_test_tfidf[:, class_aware_mask]

print("\nExperiment C — Class-aware Selection")
print("--------------------------------------")
print("Train shape:", X_train_C.shape)
print("Test shape :", X_test_C.shape)


Experiment C — Class-aware Selection
--------------------------------------
Train shape: (29821, 10000)
Test shape : (12781, 10000)


In [14]:
X_train_C_combined = hstack([
    X_train_C,
    csr_matrix(X_sent_train.values)
])

X_test_C_combined = hstack([
    X_test_C,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_C_combined.shape)
print("Final test shape :", X_test_C_combined.shape)

Final train shape: (29821, 10004)
Final test shape : (12781, 10004)


In [15]:
# ============================================================
# CELL 8: CLASS-SPECIFIC FEATURE ANALYSIS
# ============================================================

for class_id in range(n_classes):

    class_name = label_encoder.classes_[class_id]

    scores = class_scores[class_id]

    top_indices = np.argsort(scores)[::-1][:10]

    print("\n" + "=" * 60)
    print(f"CLASS: {class_name}")
    print("=" * 60)

    for idx in top_indices:
        print(
            f"{feature_names[idx]:40s} "
            f"score = {scores[idx]:.4f}"
        )


CLASS: age
school                                   score = 1935.9116
high                                     score = 1466.8608
high school                              score = 1435.7483
bullied                                  score = 1168.5133
girl                                     score = 1032.3667
bully                                    score = 951.7952
girl bullied                             score = 689.4127
bullied high                             score = 671.6403
school bully                             score = 631.7470
middle school                            score = 293.5176

CLASS: ethnicity
nigger                                   score = 1957.7339
dumb                                     score = 1790.0345
fuck                                     score = 1557.5420
dumb as                                  score = 773.7468
as                                       score = 687.4767
as nigger                                score = 669.9561
dumb nigger                       

In [16]:
# ============================================================
# CELL 9: EXPERIMENT D — GLOBAL + CLASS-AWARE
# ============================================================

global_indices = np.where(
    global_selector.get_support()
)[0]

combined_indices = np.union1d(
    global_indices,
    class_aware_indices
)

print("Global features:", len(global_indices))
print("Class-aware features:", len(class_aware_indices))
print("Union features:", len(combined_indices))

Global features: 10000
Class-aware features: 10000
Union features: 10747


In [17]:
X_train_D = X_train_tfidf[:, combined_indices]
X_test_D = X_test_tfidf[:, combined_indices]

print("\nExperiment D — Global + Class-aware")
print("-------------------------------------")
print("Train shape:", X_train_D.shape)
print("Test shape :", X_test_D.shape)


Experiment D — Global + Class-aware
-------------------------------------
Train shape: (29821, 10747)
Test shape : (12781, 10747)


In [18]:
X_train_D_combined = hstack([
    X_train_D,
    csr_matrix(X_sent_train.values)
])

X_test_D_combined = hstack([
    X_test_D,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_D_combined.shape)
print("Final test shape :", X_test_D_combined.shape)

Final train shape: (29821, 10751)
Final test shape : (12781, 10751)


In [19]:
# ============================================================
# CELL 10: EXPERIMENT SUMMARY
# ============================================================

print("=" * 70)
print("FEATURE SELECTION EXPERIMENT SUMMARY")
print("=" * 70)

print(f"A. No Selection       : {X_train_A.shape[1]} text features")
print(f"B. Global Selection   : {X_train_B.shape[1]} text features")
print(f"C. Class-aware        : {X_train_C.shape[1]} text features")
print(f"D. Global + Class     : {X_train_D.shape[1]} text features")

FEATURE SELECTION EXPERIMENT SUMMARY
A. No Selection       : 60000 text features
B. Global Selection   : 10000 text features
C. Class-aware        : 10000 text features
D. Global + Class     : 10747 text features


In [20]:
# ============================================================
# NO BALANCING + LOGISTIC REGRESSION + EVALUATION
# FIXED HYPERPARAMETERS: C = 1, max_iter = 1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LOGISTIC REGRESSION WITHOUT CLASS WEIGHTS
# ------------------------------------------------------------

lr_none = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Logistic Regression...")
print("Fixed hyperparameters: C=1, max_iter=1000, random_state=42")
print("Balancing: None")

lr_none.fit(
    X_train_A_combined,
    y_train
)

# ------------------------------------------------------------
# 3. TRAINED MODEL
# ------------------------------------------------------------

best_lr_none = lr_none

# ------------------------------------------------------------
# 4. PREDICTION
# ------------------------------------------------------------

y_pred_lr_none = best_lr_none.predict(
    X_test_A_combined
)

# ------------------------------------------------------------
# 5. EVALUATION
# ------------------------------------------------------------

lr_none_accuracy = accuracy_score(
    y_test,
    y_pred_lr_none
)

lr_none_weighted_f1 = f1_score(
    y_test,
    y_pred_lr_none,
    average='weighted'
)

lr_none_macro_f1 = f1_score(
    y_test,
    y_pred_lr_none,
    average='macro'
)

# ------------------------------------------------------------
# 6. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LOGISTIC REGRESSION")
print("==============================================")

print(f"Accuracy     : {lr_none_accuracy:.4f}")
print(f"Weighted F1  : {lr_none_weighted_f1:.4f}")
print(f"Macro F1     : {lr_none_macro_f1:.4f}")

# ------------------------------------------------------------
# 7. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_lr_none,
        labels=list(range(len(label_encoder.classes_))),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Training SA + No Balancing + Logistic Regression...
Fixed hyperparameters: C=1, max_iter=1000, random_state=42
Balancing: None

NO BALANCING + LOGISTIC REGRESSION
Accuracy     : 0.8451
Weighted F1  : 0.8451
Macro F1     : 0.8253

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.95      0.98      0.96      2388
          ethnicity       0.97      0.98      0.97      2228
             gender       0.92      0.83      0.87      2243
  not_cyberbullying       0.60      0.60      0.60      1931
other_cyberbullying       0.58      0.60      0.59      1602
           religion       0.94      0.96      0.95      2389

           accuracy                           0.85     12781
          macro avg       0.83      0.83      0.83     12781
       weighted avg       0.85      0.85      0.85     12781



In [21]:
# ============================================================
# EXPERIMENT E: 10K TF-IDF + SENTIMENT ANALYSIS
# NO BALANCING + LOGISTIC REGRESSION + EVALUATION
# FIXED HYPERPARAMETERS: C = 1, max_iter = 1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LOGISTIC REGRESSION — NO BALANCING
# ------------------------------------------------------------

lr_none_E = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN ON ORIGINAL TRAINING DATA
# ------------------------------------------------------------

print("Training Experiment E: 10K TF-IDF + Sentiment")
print("Model: Logistic Regression")
print("Balancing: None")
print("Fixed hyperparameters: C=1, max_iter=1000, random_state=42")

lr_none_E.fit(
    X_train_E_combined,
    y_train
)

# ------------------------------------------------------------
# 3. TRAINED MODEL
# ------------------------------------------------------------

best_lr_none_E = lr_none_E

# ------------------------------------------------------------
# 4. PREDICTION ON TEST DATA
# ------------------------------------------------------------

y_pred_lr_none_E = best_lr_none_E.predict(
    X_test_E_combined
)

# ------------------------------------------------------------
# 5. EVALUATION
# ------------------------------------------------------------

lr_none_E_accuracy = accuracy_score(
    y_test,
    y_pred_lr_none_E
)

lr_none_E_weighted_f1 = f1_score(
    y_test,
    y_pred_lr_none_E,
    average='weighted'
)

lr_none_E_macro_f1 = f1_score(
    y_test,
    y_pred_lr_none_E,
    average='macro'
)

# ------------------------------------------------------------
# 6. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("EXPERIMENT E: 10K TF-IDF + SENTIMENT")
print("NO BALANCING + LOGISTIC REGRESSION")
print("==============================================")

print(f"Accuracy     : {lr_none_E_accuracy:.4f}")
print(f"Weighted F1  : {lr_none_E_weighted_f1:.4f}")
print(f"Macro F1     : {lr_none_E_macro_f1:.4f}")

# ------------------------------------------------------------
# 7. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_lr_none_E,
        labels=list(range(len(label_encoder.classes_))),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Training Experiment E: 10K TF-IDF + Sentiment
Model: Logistic Regression
Balancing: None
Fixed hyperparameters: C=1, max_iter=1000, random_state=42

EXPERIMENT E: 10K TF-IDF + SENTIMENT
NO BALANCING + LOGISTIC REGRESSION
Accuracy     : 0.8436
Weighted F1  : 0.8450
Macro F1     : 0.8248

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.95      0.97      0.96      2388
          ethnicity       0.97      0.98      0.98      2228
             gender       0.92      0.83      0.87      2243
  not_cyberbullying       0.59      0.61      0.60      1931
other_cyberbullying       0.57      0.60      0.59      1602
           religion       0.95      0.95      0.95      2389

           accuracy                           0.84     12781
          macro avg       0.83      0.82      0.82     12781
       weighted avg       0.85      0.84      0.85     12781



In [22]:
# ============================================================
# NO BALANCING + LOGISTIC REGRESSION + EVALUATION
# FIXED HYPERPARAMETERS: C = 1, max_iter = 1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LOGISTIC REGRESSION WITHOUT CLASS WEIGHTS
# ------------------------------------------------------------

lr_none_B = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Logistic Regression (B)...")
print("Fixed hyperparameters: C=1, max_iter=1000, random_state=42")
print("Balancing: None")

lr_none_B.fit(
    X_train_B_combined,
    y_train
)

# ------------------------------------------------------------
# 3. TRAINED MODEL
# ------------------------------------------------------------

best_lr_none_B = lr_none_B

# ------------------------------------------------------------
# 4. PREDICTION
# ------------------------------------------------------------

y_pred_lr_none_B = best_lr_none_B.predict(
    X_test_B_combined
)

# ------------------------------------------------------------
# 5. EVALUATION
# ------------------------------------------------------------

lr_none_B_accuracy = accuracy_score(
    y_test,
    y_pred_lr_none_B
)

lr_none_B_weighted_f1 = f1_score(
    y_test,
    y_pred_lr_none_B,
    average='weighted'
)

lr_none_B_macro_f1 = f1_score(
    y_test,
    y_pred_lr_none_B,
    average='macro'
)

# ------------------------------------------------------------
# 6. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LOGISTIC REGRESSION (B)")
print("==============================================")

print(f"Accuracy     : {lr_none_B_accuracy:.4f}")
print(f"Weighted F1  : {lr_none_B_weighted_f1:.4f}")
print(f"Macro F1     : {lr_none_B_macro_f1:.4f}")

# ------------------------------------------------------------
# 7. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_lr_none_B,
        labels=list(range(len(label_encoder.classes_))),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Training SA + No Balancing + Logistic Regression (B)...
Fixed hyperparameters: C=1, max_iter=1000, random_state=42
Balancing: None

NO BALANCING + LOGISTIC REGRESSION (B)
Accuracy     : 0.8448
Weighted F1  : 0.8447
Macro F1     : 0.8249

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.94      0.97      0.96      2388
          ethnicity       0.97      0.98      0.97      2228
             gender       0.92      0.83      0.87      2243
  not_cyberbullying       0.60      0.62      0.61      1931
other_cyberbullying       0.59      0.58      0.59      1602
           religion       0.94      0.95      0.94      2389

           accuracy                           0.84     12781
          macro avg       0.83      0.82      0.82     12781
       weighted avg       0.85      0.84      0.84     12781



In [23]:
# ============================================================
# NO BALANCING + LOGISTIC REGRESSION + EVALUATION
# FIXED HYPERPARAMETERS: C = 1, max_iter = 1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LOGISTIC REGRESSION WITHOUT CLASS WEIGHTS
# ------------------------------------------------------------

lr_none_C = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Logistic Regression (C)...")
print("Fixed hyperparameters: C=1, max_iter=1000, random_state=42")
print("Balancing: None")

lr_none_C.fit(
    X_train_C_combined,
    y_train
)

# ------------------------------------------------------------
# 3. TRAINED MODEL
# ------------------------------------------------------------

best_lr_none_C = lr_none_C

# ------------------------------------------------------------
# 4. PREDICTION
# ------------------------------------------------------------

y_pred_lr_none_C = best_lr_none_C.predict(
    X_test_C_combined
)

# ------------------------------------------------------------
# 5. EVALUATION
# ------------------------------------------------------------

lr_none_C_accuracy = accuracy_score(
    y_test,
    y_pred_lr_none_C
)

lr_none_C_weighted_f1 = f1_score(
    y_test,
    y_pred_lr_none_C,
    average='weighted'
)

lr_none_C_macro_f1 = f1_score(
    y_test,
    y_pred_lr_none_C,
    average='macro'
)

# ------------------------------------------------------------
# 6. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LOGISTIC REGRESSION (C)")
print("==============================================")

print(f"Accuracy     : {lr_none_C_accuracy:.4f}")
print(f"Weighted F1  : {lr_none_C_weighted_f1:.4f}")
print(f"Macro F1     : {lr_none_C_macro_f1:.4f}")

# ------------------------------------------------------------
# 7. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_lr_none_C,
        labels=list(range(len(label_encoder.classes_))),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Training SA + No Balancing + Logistic Regression (C)...
Fixed hyperparameters: C=1, max_iter=1000, random_state=42
Balancing: None

NO BALANCING + LOGISTIC REGRESSION (C)
Accuracy     : 0.8439
Weighted F1  : 0.8438
Macro F1     : 0.8237

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.94      0.97      0.96      2388
          ethnicity       0.97      0.98      0.97      2228
             gender       0.92      0.83      0.87      2243
  not_cyberbullying       0.60      0.63      0.61      1931
other_cyberbullying       0.59      0.57      0.58      1602
           religion       0.94      0.95      0.94      2389

           accuracy                           0.84     12781
          macro avg       0.82      0.82      0.82     12781
       weighted avg       0.84      0.84      0.84     12781



In [24]:
# ============================================================
# NO BALANCING + LOGISTIC REGRESSION + EVALUATION
# FIXED HYPERPARAMETERS: C = 1, max_iter = 1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LOGISTIC REGRESSION WITHOUT CLASS WEIGHTS
# ------------------------------------------------------------

lr_none_D = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Logistic Regression (D)...")
print("Fixed hyperparameters: C=1, max_iter=1000, random_state=42")
print("Balancing: None")

lr_none_D.fit(
    X_train_D_combined,
    y_train
)

# ------------------------------------------------------------
# 3. TRAINED MODEL
# ------------------------------------------------------------

best_lr_none_D = lr_none_D

# ------------------------------------------------------------
# 4. PREDICTION
# ------------------------------------------------------------

y_pred_lr_none_D = best_lr_none_D.predict(
    X_test_D_combined
)

# ------------------------------------------------------------
# 5. EVALUATION
# ------------------------------------------------------------

lr_none_D_accuracy = accuracy_score(
    y_test,
    y_pred_lr_none_D
)

lr_none_D_weighted_f1 = f1_score(
    y_test,
    y_pred_lr_none_D,
    average='weighted'
)

lr_none_D_macro_f1 = f1_score(
    y_test,
    y_pred_lr_none_D,
    average='macro'
)

# ------------------------------------------------------------
# 6. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LOGISTIC REGRESSION (D)")
print("==============================================")

print(f"Accuracy     : {lr_none_D_accuracy:.4f}")
print(f"Weighted F1  : {lr_none_D_weighted_f1:.4f}")
print(f"Macro F1     : {lr_none_D_macro_f1:.4f}")

# ------------------------------------------------------------
# 7. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_lr_none_D,
        labels=list(range(len(label_encoder.classes_))),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Training SA + No Balancing + Logistic Regression (D)...
Fixed hyperparameters: C=1, max_iter=1000, random_state=42
Balancing: None

NO BALANCING + LOGISTIC REGRESSION (D)
Accuracy     : 0.8446
Weighted F1  : 0.8445
Macro F1     : 0.8246

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.94      0.97      0.96      2388
          ethnicity       0.97      0.98      0.97      2228
             gender       0.92      0.83      0.87      2243
  not_cyberbullying       0.60      0.62      0.61      1931
other_cyberbullying       0.59      0.58      0.59      1602
           religion       0.93      0.95      0.94      2389

           accuracy                           0.84     12781
          macro avg       0.83      0.82      0.82     12781
       weighted avg       0.85      0.84      0.84     12781

